# 🔱 VoiceBatch Studio v2.3.2 - [Ultimate Fix]
इसमें Chatterbox इंस्टॉलेशन एरर और लंबी स्क्रिप्ट दोनों का पक्का इलाज है।

In [ ]:
# @title 💤 Step 1: पक्का इंस्टॉलेशन (Fixing Build Errors)
import os, sys
from IPython.display import display, Javascript
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ इंस्टॉलेशन शुरू हो रहा है... इसमें थोड़ा समय लगेगा क्योंकि हम एरर फिक्स कर रहे हैं।")
# Chatterbox की डिपेंडेंसी एरर फिक्स करने के लिए स्टेप्स
!pip install -q --upgrade pip
!pip install -q spacy
!python -m spacy download ja_core_news_sm
!pip install -q gradio librosa soundfile coqui-tts chatterbox-tts

os.makedirs("outputs", exist_ok=True)
print("✅ सब कुछ इंस्टॉल हो गया है! अगर फिर भी 'ModuleNotFound' आए, तो ऊपर Runtime > Restart Session करें और फिर Step 2 चलाएं।")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (Auto-Split + Dual Engine)
app_code = r'''
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def strict_hindi_filter(text):
    pattern = re.compile(r'[^\u0900-\u097F\s।,?!:;0-9\[\]]')
    return pattern.sub('', text)

def studio_pro_engine(engine_type, text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    if lang == 'hi': text = strict_hindi_filter(text)
    
    out_path = 'outputs/VoiceBatch_Studio_Output.wav'
    temp_output = 'outputs/temp_part.wav'
    
    # लंबी स्क्रिप्ट को वाक्यों में तोड़ना
    sentences = re.split(r'(?<=[।?!])\s+', text)
    combined_wav = []
    
    if engine_type == 'XTTS v2 (Stable)':
        model = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
        for part in sentences:
            if len(part.strip()) < 2: continue
            model.tts_to_file(text=part, speaker_wav=audio_sample, language=lang, file_path=temp_output)
            y_p, sr = librosa.load(temp_output)
            combined_wav.extend(y_p)
    else:
        # Chatterbox Turbo Engine Fix
        from chatterbox.tts_turbo import ChatterboxTurboTTS
        model = ChatterboxTurboTTS.from_pretrained(device=device)
        wav = model.generate(text, audio_prompt_path=audio_sample)
        combined_wav = wav.cpu().numpy().squeeze()
        sr = model.sr

    y = np.array(combined_wav)
    sr = 24000 if engine_type == 'XTTS v2 (Stable)' else sr
    
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.3.2')
    with gr.Row():
        with gr.Column():
            engine = gr.Radio(['XTTS v2 (Stable)', 'Chatterbox Turbo (Fast)'], label='Select Engine', value='XTTS v2 (Stable)')
            txt = gr.Textbox(label='Script', lines=8)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en'], label='Language', value='hi')
            spd = gr.Slider(0.7, 1.4, 1.0, label="Speed")
            ptc = gr.Slider(-4, 4, 0, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate Audio ⚡', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download Output')

    btn.click(studio_pro_engine, [engine, txt, smp, spd, ptc, lng, sil], out)
demo.launch(share=True, debug=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py